# Lab: from raw Titanic rows to a saved model

Companion notebook of the lesson *Pipelines* — https://insper.github.io/ML/2026.2/classes/pipelines/

Goal: turn the raw Titanic table into one pipeline (preprocessing + a black-box k-NN classifier), fit it on the training rows, check what every step learned, predict on new raw passengers, save and reload it, and extend it with a transformer of your own. Run the cells in order, in one session.

The **fundamental part** is steps 1 to 5; the **advanced part**, steps 6 to 9. Every step starts with a **Predict**: answer it *before* running the cell. The **Simulations** ask for a small experiment of your own in the empty cell below them; the solutions are on the lesson page.

## Fundamental part — steps 1 to 5

### Step 1 — load and look

**Predict.** Which columns of a raw passenger list do you expect to have holes? Which ones could a model not use as they are?

In [ ]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula04/titanic/train.csv"
df = pd.read_csv(url)
print(df.shape)
print("missing:", df.isna().sum()[lambda s: s > 0].to_dict())
print(df[['Name', 'Sex', 'Age', 'Fare', 'Cabin', 'Embarked']].head(3).to_string())

### Step 2 — split once

**Predict.** With `stratify=y`, how close will the survival rates of the training and test rows be? And why keep in `X` the columns the model will never use, such as `Name` and `Ticket`?

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='Survived')              # all 11 raw columns, text and holes included
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(f"survival rate: train {y_train.mean():.3f} · test {y_test.mean():.3f}")

### Step 3 — the preprocessing, column by column

**Predict.** Which of the 11 raw columns will reach the model, and what happens to the ones no branch mentions? Has anything been fit when this cell ends?

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([
    ('num',  make_pipeline(SimpleImputer(strategy='median'), StandardScaler()),
             ['Age', 'SibSp', 'Parch']),
    ('fare', make_pipeline(SimpleImputer(strategy='median'),
                           FunctionTransformer(np.log1p, feature_names_out='one-to-one'),
                           StandardScaler()),
             ['Fare']),
    ('cat',  make_pipeline(SimpleImputer(strategy='most_frequent'),
                           OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
             ['Pclass', 'Sex', 'Embarked']),
])
model = Pipeline([('prep', preprocess), ('knn', KNeighborsClassifier())])

used = [c for _, _, cols in preprocess.transformers for c in cols]
for name, _, cols in preprocess.transformers:
    print(f"{name:4s} <- {cols}")
print("dropped by remainder='drop':", [c for c in X.columns if c not in used])

### Step 4 — fit on train, score on test

**Predict.** How many separate `fit` calls does the single `model.fit` trigger? Will a k-NN with its default settings beat the "always did not survive" baseline by a lot or by a little?

In [ ]:
from sklearn.dummy import DummyClassifier

model.fit(X_train, y_train)
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print(f"test accuracy · pipeline {model.score(X_test, y_test):.3f} · "
      f"always 'did not survive' {baseline.score(X_test, y_test):.3f}")

**Simulation: the same recipe without the scalers.** Build a second `ColumnTransformer` in which the `'num'` and `'fare'` branches keep only their imputer (no logarithm, no scaler), put it in a pipeline with the same `KNeighborsClassifier()`, fit it on the training rows and compare its test accuracy with the 0.782 of the pipeline. Which column dominates the distances when nothing is scaled?

In [ ]:
# your simulation here

### Step 5 — what each step learned

**Predict.** The median age of all 891 passengers is 28.0. Has the imputer inside the pipeline learned 28.0, or another number? And how many rows has the scaler seen?

In [ ]:
prep = model.named_steps['prep']
num_imputer, num_scaler = prep.named_transformers_['num']
print("Age, SibSp, Parch medians:", num_imputer.statistics_)
print("means after imputing:     ", num_scaler.mean_.round(3))
print("standard deviations:      ", num_scaler.scale_.round(3))
print("rows the scaler saw:      ", num_scaler.n_samples_seen_)
print("Age median, train only vs all 891 rows:", X_train['Age'].median(), "vs", X['Age'].median())
fare_imputer, _, fare_scaler = prep.named_transformers_['fare']
print(f"Fare: median {fare_imputer.statistics_[0]:.2f} · mean of log1p(Fare) {fare_scaler.mean_[0]:.3f}")
print("categories:", [c.tolist() for c in prep.named_transformers_['cat'][-1].categories_])

**Simulation: the preprocessing fit on all 891 passengers.** Reproduce the leak of section 4 on the Titanic: fit a copy of `preprocess` on all of `X` (use `clone` from `sklearn.base`), fit a `KNeighborsClassifier()` on the transformed training rows only, and score it on the transformed test rows. What median age does the copy learn? Does the test accuracy change? Then fit a copy of the whole `model` on all 891 rows and score it on the test rows.

In [ ]:
# your simulation here

**To close the fundamental part.** In one sentence each: (1) why is the bug that re-fits a scaler on the test set the most dangerous of the three? (2) What does `model.predict` do to each transformer, and why does that make the size of the batch irrelevant? (3) Why did fitting the scaler on all rows barely change the score, while the target encoder made it look about three times better?

### Your turn (fundamental)


At home:

1. **By hand.** A column has training values 10, NaN, 30, 50 and new values NaN, 70. Compute the median, mean and standard deviation that `SimpleImputer(strategy='median')` followed by `StandardScaler()` learn, and the two transformed new values. Check with a pipeline. Then compute what `fit_transform` on the two new rows would give.
2. **Bug 1 with another scaler.** Repeat the regional-batch experiment of bug 1 with `MinMaxScaler` instead of `StandardScaler`, in both the model and the bug. Is the damage larger or smaller? Which statistic of a batch does min–max re-learn, and why is it more fragile than a mean?
3. **The size of a leak.** Repeat the target-encoding experiment of section 4 with zones rounded to one decimal (about 10 km) instead of two. How many zones have a single district now? Does the gap between the leaky and the honest MAE grow or shrink, and why?
4. **Break it on purpose.** Predict on `X_test` with its columns in reverse order; on `X_test.to_numpy()`; and on `X_test.drop(columns='Cabin')`. Which of the three work, which fail, and what in the pipeline explains each result?

## Advanced part — steps 6 to 9

### Step 6 — what the model sees

**Predict.** How many columns reach the model: the 7 columns the branches use, or more? Will a third-class man have a 1 in one `Pclass` column or in several?

In [ ]:
names = prep.get_feature_names_out()
print(len(names), "columns reach the model:", list(names))
prep.set_output(transform='pandas')
print(prep.transform(X_test.head(3)).round(2).T)

### Step 7 — new passengers, raw

**Predict.** One new passenger boarded at a port the training data never saw, and another has no age. Will the pipeline crash, or answer? What number will replace the missing age?

In [ ]:
new = pd.DataFrame({
    'PassengerId': [901, 902, 903], 'Pclass': [3, 1, 2],
    'Name': ['Doe, Mr. John', 'Roe, Mrs. Jane', 'Poe, Miss. Ann'],
    'Sex': ['male', 'female', 'female'], 'Age': [25, np.nan, 30],
    'SibSp': [0, 1, 0], 'Parch': [0, 0, 0], 'Ticket': ['A1', 'B2', 'C3'],
    'Fare': [8.05, 80.0, 15.0], 'Cabin': [np.nan, 'C85', np.nan],
    'Embarked': ['S', 'C', 'B'],             # 'B' (Belfast) never appears in the training data
})
print(prep.transform(new)[['num__Age', 'cat__Embarked_C', 'cat__Embarked_Q', 'cat__Embarked_S']].round(2))
print("P(survived):", model.predict_proba(new)[:, 1].round(2), "· prediction:", model.predict(new))

**Simulation: an encoder that refuses new categories.** Make a copy of `model` (with `clone`) whose one-hot encoder of the `'cat'` branch has `handle_unknown='error'`; the path is `prep__cat__onehotencoder__handle_unknown`, set with `set_params`. Fit it on the training rows and predict the first two new passengers, then all three. What happens, and what would it mean for a service that answers one request at a time?

In [ ]:
# your simulation here

### Step 8 — save, load, compare

**Predict.** Will the saved file be a few kilobytes or hundreds? Will the loaded model give exactly the same probabilities, or only approximately?

In [ ]:
import os
import joblib

joblib.dump(model, 'titanic-knn.joblib')
loaded = joblib.load('titanic-knn.joblib')
print(f"{os.path.getsize('titanic-knn.joblib') / 1e3:.0f} kB")
print("same test predictions:", np.array_equal(loaded.predict(X_test), model.predict(X_test)))
print("same probabilities for the new rows:",
      np.array_equal(loaded.predict_proba(new), model.predict_proba(new)))

### Step 9 — a transformer of your own: the title

**Predict.** Should the list of "frequent" titles be learned on the training rows, or can it be written by hand from all 891 passengers? Will the title improve the accuracy, and are 179 test passengers enough to be sure?

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class TitleExtractor(TransformerMixin, BaseEstimator):
    """Name -> title ('Mr', 'Mrs', ...); titles rarer than min_count in TRAIN become 'Rare'."""
    def __init__(self, min_count=10):
        self.min_count = min_count

    def _titles(self, X):
        names = pd.Series(np.asarray(X).ravel())
        return names.str.extract(r',\s*([^.]+)\.')[0].str.strip()

    def fit(self, X, y=None):
        counts = self._titles(X).value_counts()
        self.titles_ = sorted(counts[counts >= self.min_count].index)   # learned on train
        return self

    def transform(self, X):
        t = self._titles(X)
        return t.where(t.isin(self.titles_), 'Rare').to_frame('title')

    def get_feature_names_out(self, input_features=None):
        return np.array(['title'])

prep2 = ColumnTransformer(preprocess.transformers + [
    ('title', make_pipeline(TitleExtractor(), OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
     ['Name'])])
model2 = Pipeline([('prep', prep2), ('knn', KNeighborsClassifier())]).fit(X_train, y_train)
print("titles learned on train:", model2['prep'].named_transformers_['title'][0].titles_)
print("titles of the new rows: ", TitleExtractor().fit(X_train[['Name']]).transform(new[['Name']])['title'].to_list())
print(f"test accuracy · without title {model.score(X_test, y_test):.3f} · with title {model2.score(X_test, y_test):.3f}")

**Simulation: how rare is "Rare"?** `min_count` is a hyperparameter of `TitleExtractor`, so the pipeline reaches it as `prep__title__titleextractor__min_count`. For `min_count` in 1, 5, 10 and 40, fit a copy of `model2` (with `clone` and `set_params`) and print the titles it learned and its test accuracy. Which titles survive each threshold, and does the accuracy say which threshold is best?

In [ ]:
# your simulation here

**To close the lesson.** In one sentence each: (1) why save the whole pipeline and not only the trained classifier? (2) When must a step of your own be a class rather than a `FunctionTransformer`? (3) Why does a search that tries settings through `set_params` never let the scoring rows into the preprocessing?

### Your turn (advanced)


At home, with more time:

1. **Ratios as a step.** Rooms per household, bedrooms per room and people per household are more meaningful than raw counts. Write them as a `FunctionTransformer` (use a function that receives a DataFrame and returns one with the three new columns, and `feature_names_out` as a function that returns their names), put it in the numeric branch of section 3, and compare the test MAE with the lesson's \$41,552.
2. **The deck.** The first letter of `Cabin` is the deck (A to G, T), and a missing cabin is information too (mostly third class). Add a branch that turns `Cabin` into the deck, with `'U'` for missing, followed by one-hot encoding. Write the conversion first as a `lambda` inside a `FunctionTransformer`, fit, and try `joblib.dump` on the model. What happens, and why does a function defined with `def` fix it?
3. **Choose `k` honestly.** Carve a validation split out of `X_train` (25% of it, stratified). For `k` in 1, 5, 15, 45, use `model.set_params(knn__n_neighbors=k)`, fit on the rest of the training rows and score on the validation rows. Pick the best `k`, refit on all of `X_train`, and only then look at the test accuracy, once. (The [cross-validation](https://insper.github.io/ML/2026.2/classes/validation/) lesson will make this choice more robust.)
4. **Clip the fare.** Replace `log1p` in the `'fare'` branch with the lesson's `Clipper` (percentiles 1 and 99). Print its `upper_` and compare it with the largest fare in the training rows. Does the test accuracy change? Would you expect k-NN to be more sensitive to the fare's outliers than a model that only compares a value to a threshold?